# Reproduce Manuscript Figures

This notebook reproduces the three main figures for *Hypsographic Demography: Age and Change in Population by Altitude* from Dataset S1. It does not run the full gridded extraction workflow and does not generate exploratory figures.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from PIL import Image
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'data' / 'dataset_s1_hypsographic_demography.csv').exists():
    ROOT = Path('/Users/f6z/Documents/projects/montana_state/hypsographic_demography')

DATA = ROOT / 'data'
OUT = ROOT / 'outputs'
FIG = OUT / 'figures'
TAB = OUT / 'tables'
for d in [FIG, TAB]:
    d.mkdir(parents=True, exist_ok=True)

ELEVATION_ORDER = ['<100 m', '100-499 m', '500-1499 m', '1500-2499 m', '2500-3499 m', '>=3500 m']
SETTLEMENT_ORDER = ['Low-density rural', 'Rural cluster', 'Peri-urban', 'Semi-dense urban', 'Dense urban', 'Urban centre']
ELEVATION_LABELS = {'<100 m': '<100 m', '100-499 m': '100–499 m', '500-1499 m': '500–1,499 m', '1500-2499 m': '1,500–2,499 m', '2500-3499 m': '2,500–3,499 m', '>=3500 m': '≥3,500 m'}
AGE_COLORS = {'0-14': '#3B9AB2', '15-64': '#E1AF00', '65+': '#F21A00'}
PNAS_FULL_WIDTH_IN = 17.8 / 2.54
GROWTH_COLOR_LIMIT = 25

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
if 'Helvetica' in available_fonts:
    FIGURE_FONT = 'Helvetica'
elif 'Arial' in available_fonts:
    FIGURE_FONT = 'Arial'
else:
    FIGURE_FONT = 'DejaVu Sans'

mpl.rcParams.update({
    'font.family': FIGURE_FONT,
    'font.size': 6.6,
    'axes.titlesize': 7.4,
    'axes.labelsize': 6.8,
    'xtick.labelsize': 6.0,
    'ytick.labelsize': 6.0,
    'legend.fontsize': 6.0,
    'axes.linewidth': 0.55,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'savefig.facecolor': 'white',
})

def pct(x, digits=1):
    return f'{100 * x:.{digits}f}%'

def bill(x, digits=2):
    return f'{x / 1e9:.{digits}f} billion'

def mill(x, digits=1):
    return f'{x / 1e6:.{digits}f} million'

def savefig(name, fig):
    for ext in ['png', 'pdf']:
        path = FIG / f'{name}.{ext}'
        fig.savefig(path, dpi=600 if ext == 'png' else None, bbox_inches='tight', facecolor='white')
        if ext == 'png':
            img = Image.open(path)
            if img.mode != 'RGB':
                img.convert('RGB').save(path)

def style_axes(ax, grid_axis='x'):
    ax.set_facecolor('white')
    for spine in ax.spines.values():
        spine.set_linewidth(0.55)
        spine.set_color('0.25')
    if grid_axis:
        ax.grid(axis=grid_axis, color='0.90', linewidth=0.45)
        ax.set_axisbelow(True)

def panel_label(ax, label):
    ax.text(-0.07, 1.045, label, transform=ax.transAxes, fontweight='bold', va='bottom', ha='left', fontsize=8.8)

print('Repo root:', ROOT)
print('Figure font:', FIGURE_FONT)


## Read Dataset S1

In [ ]:
s1 = pd.read_csv(DATA / 'dataset_s1_hypsographic_demography.csv')
print(f'Dataset S1 rows: {len(s1):,}')
print('Column definitions are documented in data/README.md.')


## Rebuild Figure Tables

In [ ]:
# Figure 1 table.
f1 = s1[s1['table_name'].eq('figure1_elevation_summary')].copy()
fig1_rows = []
for elev in ELEVATION_ORDER:
    sub = f1[f1['elevation_group'].eq(elev)]
    row = {'elevation_group': elev}
    row['population_total_start'] = sub[(sub['metric'].eq('population')) & (sub['year'].eq(2015))]['value'].sum()
    row['population_total'] = sub[(sub['metric'].eq('population')) & (sub['year'].eq(2025)) & (sub['age_group'].isna())]['value'].sum()
    row['absolute_change'] = row['population_total'] - row['population_total_start']
    row['growth_2015_2025_percent'] = sub[sub['metric'].eq('population_growth')]['value'].sum()
    for age in ['0-14', '15-64', '65+']:
        age_sub = sub[sub['age_group'].eq(age)]
        row[f'{age}_population'] = age_sub[age_sub['metric'].eq('population')]['value'].sum()
        row[f'{age}_share_percent'] = age_sub[age_sub['metric'].eq('population_share')]['value'].sum()
    row['population_2015_billions'] = row['population_total_start'] / 1e9
    row['population_2025_billions'] = row['population_total'] / 1e9
    row['youth_population'] = row['0-14_population']
    row['working_age_population'] = row['15-64_population']
    row['old_age_population'] = row['65+_population']
    row['youth_share_percent'] = row['0-14_share_percent']
    row['working_age_share_percent'] = row['15-64_share_percent']
    row['old_age_share_percent'] = row['65+_share_percent']
    fig1_rows.append(row)
main_fig01_data = pd.DataFrame(fig1_rows)
global_pop_2025 = main_fig01_data['population_total'].sum()

main_fig01_data.to_csv(TAB / 'main_fig01_data.csv', index=False)

# Figure 1C age-contribution table.
f1c = s1[s1['table_name'].eq('figure1_age_contribution')].copy()
fig1c_rows = []
for elev in ELEVATION_ORDER:
    sub = f1c[f1c['elevation_group'].eq(elev)]
    denominator = sub[(sub['metric'].eq('population_denominator')) & (sub['year'].eq(2015))]['value'].sum()
    total_growth = sub[(sub['metric'].eq('population_growth')) & (sub['interval'].eq('2015-2025'))]['value'].sum()
    for age in ['0-14', '15-64', '65+']:
        age_sub = sub[sub['age_group'].eq(age)]
        pop_2015 = age_sub[(age_sub['metric'].eq('population')) & (age_sub['year'].eq(2015))]['value'].sum()
        pop_2025 = age_sub[(age_sub['metric'].eq('population')) & (age_sub['year'].eq(2025))]['value'].sum()
        absolute_change = age_sub[(age_sub['metric'].eq('absolute_change')) & (age_sub['interval'].eq('2015-2025'))]['value'].sum()
        contribution = age_sub[(age_sub['metric'].eq('contribution_to_total_growth')) & (age_sub['interval'].eq('2015-2025'))]['value'].sum()
        fig1c_rows.append({
            'elevation_group': elev,
            'age_group': age,
            'population_2015_ageclass': pop_2015,
            'population_2025_ageclass': pop_2025,
            'absolute_change_ageclass': absolute_change,
            'population_2015_total': denominator,
            'contribution_to_total_growth_pp': contribution,
            'total_growth_percent': total_growth,
        })
fig1_age_contribution_values = pd.DataFrame(fig1c_rows)
fig1_age_contribution_values['sum_contribution_pp'] = fig1_age_contribution_values.groupby('elevation_group')['contribution_to_total_growth_pp'].transform('sum')
fig1_age_contribution_values['sum_minus_total_growth_pp'] = fig1_age_contribution_values['sum_contribution_pp'] - fig1_age_contribution_values['total_growth_percent']
fig1_age_contribution_values.to_csv(TAB / 'fig01_age_contribution_values.csv', index=False)

# Figure 2 table.
f2 = s1[s1['table_name'].eq('figure2_elevation_settlement')].copy()
fig2_rows = []
for elev in ELEVATION_ORDER:
    for sett in SETTLEMENT_ORDER:
        sub = f2[f2['elevation_group'].eq(elev) & f2['settlement_class'].eq(sett)]
        fig2_rows.append({
            'elevation_group': elev,
            'settlement_class': sett,
            'pop_2015_static2025': sub[(sub['metric'].eq('population')) & (sub['year'].eq(2015))]['value'].sum(),
            'pop_2025_static2025': sub[(sub['metric'].eq('population')) & (sub['year'].eq(2025)) & (sub['age_group'].isna())]['value'].sum(),
            'youth_population': sub[(sub['metric'].eq('population')) & (sub['age_group'].eq('0-14'))]['value'].sum(),
            'youth_share_percent': sub[(sub['metric'].eq('population_share')) & (sub['age_group'].eq('0-14'))]['value'].sum(),
            'male_share_percent': sub[sub['metric'].eq('male_share')]['value'].sum(),
            'growth_static2025_percent': sub[sub['metric'].eq('population_growth')]['value'].sum(),
        })
main_fig02_static = pd.DataFrame(fig2_rows)
main_fig02_static['youth_share'] = main_fig02_static['youth_share_percent'] / 100
main_fig02_static['male_share'] = main_fig02_static['male_share_percent'] / 100
main_fig02_static['growth_static2025'] = main_fig02_static['growth_static2025_percent'] / 100
main_fig02_static['pop_2025_change_static2025'] = main_fig02_static['pop_2025_static2025']
main_fig02_static['population_2025_millions'] = main_fig02_static['pop_2025_static2025'] / 1e6
main_fig02_static.to_csv(TAB / 'main_fig02_static2025_data.csv', index=False)

# Dynamic/static comparison table.
comparison = s1[s1['table_name'].eq('dynamic_static_growth_comparison')].copy()
comparison_rows = []
for elev in ELEVATION_ORDER:
    for sett in SETTLEMENT_ORDER:
        sub = comparison[comparison['elevation_group'].eq(elev) & comparison['settlement_class'].eq(sett)]
        comparison_rows.append({
            'elevation_group': elev,
            'settlement_class': sett,
            'pop_2015_static2025': sub[(sub['metric'].eq('population_denominator')) & (sub['attribution_scenario'].eq('static_2025'))]['value'].sum(),
            'pop_2025_static2025': main_fig02_static.loc[(main_fig02_static['elevation_group'].eq(elev)) & (main_fig02_static['settlement_class'].eq(sett)), 'pop_2025_static2025'].sum(),
            'growth_static2025': sub[(sub['metric'].eq('population_growth')) & (sub['attribution_scenario'].eq('static_2025'))]['value'].sum(),
            'growth_dynamic': sub[(sub['metric'].eq('population_growth')) & (sub['attribution_scenario'].eq('dynamic'))]['value'].sum(),
            'difference_pp': sub[(sub['metric'].eq('growth_difference')) & (sub['attribution_scenario'].eq('dynamic_minus_static_2025'))]['value'].sum(),
        })
comparison_table = pd.DataFrame(comparison_rows)
comparison_table.to_csv(TAB / 'dynamic_vs_static2025_fig02_comparison.csv', index=False)

display(main_fig01_data)
display(main_fig02_static.head(12))


## Main Figure 1

Figure 1C decomposes modeled 2015-2025 population growth by broad age class. Segment widths show each age class's contribution, in percentage points, to total population growth within each elevation group; labels show total modeled growth.

In [ ]:
fig1_age_contrib = pd.read_csv(TAB / 'fig01_age_contribution_values.csv')
contrib_pivot = (
    fig1_age_contrib
    .pivot(index='elevation_group', columns='age_group', values='contribution_to_total_growth_pp')
    .reindex(index=ELEVATION_ORDER, columns=['0-14', '15-64', '65+'])
)
contrib_check = (
    fig1_age_contrib.groupby('elevation_group', as_index=False)
    .agg(sum_contribution_pp=('contribution_to_total_growth_pp', 'sum'), total_growth_percent=('total_growth_percent', 'first'))
)
contrib_check['difference_pp'] = contrib_check['sum_contribution_pp'] - contrib_check['total_growth_percent']
assert np.allclose(contrib_check['difference_pp'], 0, atol=1e-8), 'Figure 1C age contributions do not sum to total growth.'

y = np.arange(len(main_fig01_data))
labels = [ELEVATION_LABELS[e] for e in main_fig01_data['elevation_group'].astype(str)]
fig, axes = plt.subplots(
    1,
    3,
    figsize=(PNAS_FULL_WIDTH_IN, 2.95),
    sharey=True,
    gridspec_kw={'width_ratios': [1.10, 1.00, 1.28]},
)
fig.subplots_adjust(left=0.12, right=0.955, top=0.82, bottom=0.31, wspace=0.14)
h = 0.34
import matplotlib.patheffects as pe
FIG1_YEAR_COLORS = {'2015': '#C6CDF7', '2025': '#5F7FC8'}
FIG1_AGE_COLORS = {'0-14': '#3B9AB2', '15-64': '#E1AF00', '65+': '#E85D3F'}

def legend_text_color(hex_color, label=None):
    if label in {'2025', '0-14', '65+'}:
        return 'white'
    rgb = mpl.colors.to_rgb(hex_color)
    luminance = 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]
    return 'white' if luminance < 0.52 else '0.12'

def draw_box_legend(fig, *, items, center_x, y0, box_width=0.070, box_height=0.046, gap=0.012):
    total_width = len(items) * box_width + (len(items) - 1) * gap
    x0 = center_x - total_width / 2
    for i, (label, color) in enumerate(items):
        x = x0 + i * (box_width + gap)
        rect = mpl.patches.Rectangle(
            (x, y0),
            box_width,
            box_height,
            transform=fig.transFigure,
            facecolor=color,
            edgecolor='white',
            linewidth=0.55,
            clip_on=False,
        )
        fig.add_artist(rect)
        fig.text(
            x + box_width / 2,
            y0 + box_height / 2,
            label,
            ha='center',
            va='center',
            fontsize=6.4,
            color=legend_text_color(color, label),
            fontweight='bold' if label in {'2025', '0-14', '65+'} else 'normal',
        )

def panel_label_corner(ax, label):
    ax.text(-0.095, 1.065, label, transform=ax.transAxes, fontweight='bold', va='center', ha='left', fontsize=8.8)


ax = axes[0]
xmin = 0.008
bar2015 = ax.barh(
    y - h/2,
    main_fig01_data['population_2015_billions'] - xmin,
    left=xmin,
    height=h,
    color=FIG1_YEAR_COLORS['2015'],
    edgecolor='white',
    linewidth=0.45,
    label='2015',
)
bar2025 = ax.barh(
    y + h/2,
    main_fig01_data['population_2025_billions'] - xmin,
    left=xmin,
    height=h,
    color=FIG1_YEAR_COLORS['2025'],
    edgecolor='white',
    linewidth=0.45,
    label='2025',
)
ax.set_xscale('log')
ax.set_xlim(xmin, 5.0)
ax.set_xticks([0.01, 0.1, 1, 5])
ax.set_xticklabels(['10M', '100M', '1B', '5B'])
ax.set_yticks(y, labels)
ax.set_xlabel('Population, log scale')
ax.set_ylabel('Elevation group')
ax.set_title('Population by elevation')
draw_box_legend(
    fig,
    items=[('2015', FIG1_YEAR_COLORS['2015']), ('2025', FIG1_YEAR_COLORS['2025'])],
    center_x=(axes[0].get_position().x0 + axes[0].get_position().x1) / 2,
    y0=0.120,
)
style_axes(ax, grid_axis='x')
panel_label_corner(ax, 'A')

ax = axes[1]
left = np.zeros(len(main_fig01_data))
for col, lab in [('youth_share_percent', '0-14'), ('working_age_share_percent', '15-64'), ('old_age_share_percent', '65+')]:
    vals = main_fig01_data[col].to_numpy()
    ax.barh(y, vals, left=left, color=FIG1_AGE_COLORS[lab], edgecolor='white', linewidth=0.45, label=lab)
    left += vals
ax.set_xlim(0, 100)
ax.set_xlabel('Population share (%)')
ax.set_title('Age composition, 2025')
style_axes(ax, grid_axis='x')
panel_label_corner(ax, 'B')

ax = axes[2]
pos_left = np.zeros(len(contrib_pivot))
neg_left = np.zeros(len(contrib_pivot))
for age in ['0-14', '15-64', '65+']:
    vals = contrib_pivot[age].to_numpy()
    left = np.where(vals >= 0, pos_left, neg_left)
    ax.barh(y, vals, left=left, color=FIG1_AGE_COLORS[age], edgecolor='white', linewidth=0.45, label=age)
    pos_left += np.where(vals > 0, vals, 0)
    neg_left += np.where(vals < 0, vals, 0)
ax.axvline(0, color='0.28', linewidth=0.7)
growth_label_x = 20.4
ax.set_xlim(min(-2, np.nanmin(neg_left) - 0.5), 24.0)
ax.text(
    growth_label_x,
    0.975,
    'Total change',
    transform=ax.get_xaxis_transform(),
    ha='center',
    va='top',
    fontsize=6.0,
    fontweight='bold',
    color='0.12',
    clip_on=False,
    path_effects=[pe.withStroke(linewidth=1.1, foreground='white')],
)
for i, elev in enumerate(contrib_pivot.index):
    total_growth = fig1_age_contrib.loc[fig1_age_contrib['elevation_group'].eq(elev), 'total_growth_percent'].iloc[0]
    ax.text(growth_label_x, y[i], f'{total_growth:+.1f}%', va='center', ha='center', fontsize=5.9, color='0.12', path_effects=[pe.withStroke(linewidth=1.2, foreground='white')])
ax.set_xlabel('Contribution to total change (percentage points)')
ax.set_title('Population change by age group, 2015–2025')
style_axes(ax, grid_axis='x')
panel_label_corner(ax, 'C')

draw_box_legend(
    fig,
    items=[(a, FIG1_AGE_COLORS[a]) for a in ['0-14', '15-64', '65+']],
    center_x=(axes[1].get_position().x0 + axes[2].get_position().x1) / 2,
    y0=0.120,
)

savefig('fig1_hypsographic_summary', fig)
plt.show()

## Main Figure 2

In [ ]:
matrix_elev_order = list(reversed(ELEVATION_ORDER))
pop_mat = main_fig02_static.pivot(index='elevation_group', columns='settlement_class', values='population_2025_millions').reindex(index=matrix_elev_order, columns=SETTLEMENT_ORDER)
youth_mat = main_fig02_static.pivot(index='elevation_group', columns='settlement_class', values='youth_share_percent').reindex(index=matrix_elev_order, columns=SETTLEMENT_ORDER)
growth_mat = main_fig02_static.pivot(index='elevation_group', columns='settlement_class', values='growth_static2025_percent').reindex(index=matrix_elev_order, columns=SETTLEMENT_ORDER)

def annotate_matrix(ax, data, cmap, norm, fmt='{:.1f}', fontsize=5.1):
    arr = np.asarray(data, dtype=float)
    cmap_obj = mpl.colormaps[cmap] if isinstance(cmap, str) else cmap
    if not np.isfinite(arr).any():
        return
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            val = arr[i, j]
            if np.isfinite(val):
                r, g, b, _ = cmap_obj(norm(val))
                luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
                color = 'white' if luminance < 0.50 else '0.12'
                ax.text(j, i, fmt.format(val), ha='center', va='center', fontsize=fontsize, color=color, fontweight='normal')

# Sequential growth scale: 0 to observed max, no clipping/thresholding.
population_cmap = mpl.colors.LinearSegmentedColormap.from_list(
    'population_lavender_purple',
    ['#F1EEF6', '#D7D4EA', '#A6A1CF', '#756BB1', '#54278F'],
)
growth_cmap = mpl.cm.Oranges.copy()
growth_norm = mpl.colors.Normalize(vmin=0, vmax=25)
growth_ticks = [0, 5, 10, 15, 20, 25]

fig, axes = plt.subplots(1, 3, figsize=(PNAS_FULL_WIDTH_IN, 2.95), sharey=True)
fig.subplots_adjust(left=0.105, right=0.985, top=0.82, bottom=0.34, wspace=0.17)

panels = [
    (
        axes[0],
        pop_mat,
        'Population, 2025',
        population_cmap,
        mpl.colors.PowerNorm(
            gamma=0.45,
            vmin=0,
            vmax=2000
        ),
        '{:.1f}',
        'Population (millions)',
        [1, 10, 100, 1000, 2000],
        'neither',
        False
    ),
    (
        axes[1],
        youth_mat,
        'Youth share, 2025',
        'summer_r',
        mpl.colors.Normalize(vmin=20, vmax=35),
        '{:.1f}',
        'Youth share (%)',
        [20, 25, 30, 35],
        'neither',
        False
    ),
    (
        axes[2],
        growth_mat,
        'Population change, 2015-2025',
        growth_cmap,
        growth_norm,
        '{:.1f}',
        'Growth (%)',
        growth_ticks,
        'neither',
        False
    ),
]

for idx, (ax, data, title, cmap, norm, fmt, cbar_label, ticks, extend, is_growth) in enumerate(panels):
    im = ax.imshow(data, cmap=cmap, norm=norm, aspect='equal')
    annotate_matrix(ax, data, cmap=cmap, norm=norm, fmt=fmt)

    ax.set_title(title)
    ax.set_xticks(
        np.arange(len(SETTLEMENT_ORDER)),
        SETTLEMENT_ORDER,
        rotation=36,
        ha='right',
        rotation_mode='anchor'
    )
    ax.set_yticks(np.arange(len(matrix_elev_order)), [ELEVATION_LABELS[e] for e in matrix_elev_order])
    ax.tick_params(axis='x', length=0, labelsize=5.4)
    ax.tick_params(axis='y', length=0, labelleft=(idx == 0), labelsize=5.8)
    ax.set_xticks(np.arange(-0.5, len(SETTLEMENT_ORDER), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(matrix_elev_order), 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.45)
    ax.tick_params(which='minor', bottom=False, left=False)

    for spine in ax.spines.values():
        spine.set_linewidth(0.55)
        spine.set_color('0.38')

    if idx == 0:
        ax.set_ylabel('Elevation group')

    cbar = fig.colorbar(im, ax=ax, fraction=0.050, pad=0.030, ticks=ticks, extend=extend)
    cbar.ax.tick_params(labelsize=5.6, length=2)
    cbar.outline.set_linewidth(0.45)
    cbar.set_label(cbar_label, fontsize=5.9)

    panel_label(ax, chr(ord('A') + idx))

savefig('fig2_elevation_settlement_stats', fig)
plt.show()


## Main Figure 3

Figure 3 maps 2015-2025 population change across inhabited highland elevation-zone polygons (1,500-3,500 m).


In [ ]:
import runpy

runpy.run_path(str(ROOT / 'processing' / '06_plot_fig3_highland_change_map.py'), run_name='__main__')
display(Image.open(FIG / 'fig3_highland_change_map.png'))


## Key Manuscript Values

In [ ]:
below100 = main_fig01_data.loc[main_fig01_data['elevation_group'].eq('<100 m'), 'population_total'].sum()
below500 = main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['<100 m', '100-499 m']), 'population_total'].sum()
highland_1500_3500 = main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['1500-2499 m', '2500-3499 m']), 'population_total'].sum()
highland_3500 = main_fig01_data.loc[main_fig01_data['elevation_group'].eq('>=3500 m'), 'population_total'].sum()
youth_below500 = main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['<100 m', '100-499 m']), 'youth_population'].sum() / below500
youth_highland = main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['1500-2499 m', '2500-3499 m']), 'youth_population'].sum() / highland_1500_3500
growth_below500 = main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['<100 m', '100-499 m']), 'absolute_change'].sum() / main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['<100 m', '100-499 m']), 'population_total_start'].sum()
growth_highland = main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['1500-2499 m', '2500-3499 m']), 'absolute_change'].sum() / main_fig01_data.loc[main_fig01_data['elevation_group'].isin(['1500-2499 m', '2500-3499 m']), 'population_total_start'].sum()
strongest_pop = main_fig02_static.loc[main_fig02_static['pop_2025_static2025'].idxmax()]
strongest_youth = main_fig02_static.loc[main_fig02_static['youth_share_percent'].idxmax()]
strongest_growth = main_fig02_static.loc[main_fig02_static['growth_static2025_percent'].idxmax()]

integer_pop_2025 = pd.read_parquet(DATA / 'global_integer_elevation_age_sex_2015_2025.parquet')
integer_pop_2025 = integer_pop_2025[
    integer_pop_2025['geography_level'].astype(str).str.lower().eq('global')
    & integer_pop_2025['continent'].astype(str).str.lower().eq('global')
].copy()
integer_pop_2025['elevation_m'] = pd.to_numeric(integer_pop_2025['elevation_m'], errors='coerce')
integer_pop_2025['population_count'] = pd.to_numeric(integer_pop_2025['population_count'], errors='coerce').fillna(0)
integer_pop_2025 = integer_pop_2025.dropna(subset=['elevation_m'])

elev_cdf = (
    integer_pop_2025.groupby('elevation_m', as_index=False)['population_count']
    .sum()
    .sort_values('elevation_m')
)
elev_cdf['cum_pop'] = elev_cdf['population_count'].cumsum()
population_weighted_median_elevation_2025 = float(
    elev_cdf.loc[elev_cdf['cum_pop'].ge(elev_cdf['population_count'].sum() / 2), 'elevation_m'].iloc[0]
)

below150 = integer_pop_2025.loc[integer_pop_2025['elevation_m'] <= 150, 'population_count'].sum()
below150_share = below150 / integer_pop_2025['population_count'].sum()
age_totals_2025 = integer_pop_2025.groupby('broad_age_group')['population_count'].sum()
age_below150_2025 = integer_pop_2025[integer_pop_2025['elevation_m'] <= 150].groupby('broad_age_group')['population_count'].sum()
below150_age_shares = age_below150_2025 / age_totals_2025

key_values = pd.DataFrame([
    {'metric': 'population below 500 m, 2025', 'value': below500, 'unit': 'people', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': f'{bill(below500)}; {pct(below500 / global_pop_2025)} of global population'},
    {'metric': 'population below 100 m, 2025', 'value': below100, 'unit': 'people', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': bill(below100)},
    {'metric': 'population below 150 m, 2025', 'value': below150, 'unit': 'people', 'source_table': 'data/global_integer_elevation_age_sex_2015_2025.parquet', 'note': f'{bill(below150)}; {pct(below150_share)} of global population'},
    {'metric': 'population-weighted median elevation, 2025', 'value': population_weighted_median_elevation_2025, 'unit': 'meters', 'source_table': 'data/global_integer_elevation_age_sex_2015_2025.parquet', 'note': f'{population_weighted_median_elevation_2025:.0f} m'},
    {'metric': 'share of young population below 150 m, 2025', 'value': below150_age_shares['young_0_14'], 'unit': 'share', 'source_table': 'outputs/tables/population_by_elevation_threshold_age_summary.csv', 'note': pct(below150_age_shares['young_0_14'])},
    {'metric': 'share of working-age population below 150 m, 2025', 'value': below150_age_shares['working_age_15_64'], 'unit': 'share', 'source_table': 'outputs/tables/population_by_elevation_threshold_age_summary.csv', 'note': pct(below150_age_shares['working_age_15_64'])},
    {'metric': 'share of old-age population below 150 m, 2025', 'value': below150_age_shares['old_age_65_plus'], 'unit': 'share', 'source_table': 'outputs/tables/population_by_elevation_threshold_age_summary.csv', 'note': pct(below150_age_shares['old_age_65_plus'])},
    {'metric': 'population 1500-3500 m, 2025', 'value': highland_1500_3500, 'unit': 'people', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': bill(highland_1500_3500)},
    {'metric': 'youth share below 500 m, 2025', 'value': youth_below500, 'unit': 'share', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': pct(youth_below500)},
    {'metric': 'youth share 1500-3500 m, 2025', 'value': youth_highland, 'unit': 'share', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': pct(youth_highland)},
    {'metric': 'growth below 500 m, 2015-2025', 'value': growth_below500, 'unit': 'share', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': pct(growth_below500)},
    {'metric': 'growth 1500-3500 m, 2015-2025', 'value': growth_highland, 'unit': 'share', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': pct(growth_highland)},
    {'metric': 'population >=3500 m, 2025', 'value': highland_3500, 'unit': 'people', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': f'{mill(highland_3500)}; {pct(highland_3500 / global_pop_2025)} of global population'},
    {'metric': 'strongest Figure 2 population cell', 'value': strongest_pop['pop_2025_static2025'], 'unit': 'people', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': f'{strongest_pop.elevation_group} / {strongest_pop.settlement_class}'},
    {'metric': 'strongest Figure 2 youth-share cell', 'value': strongest_youth['youth_share'], 'unit': 'share', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': f'{strongest_youth.elevation_group} / {strongest_youth.settlement_class}; {pct(strongest_youth.youth_share)}'},
    {'metric': 'strongest Figure 2 growth cell', 'value': strongest_growth['growth_static2025'], 'unit': 'share', 'source_table': 'data/dataset_s1_hypsographic_demography.csv', 'note': f'{strongest_growth.elevation_group} / {strongest_growth.settlement_class}; {pct(strongest_growth.growth_static2025)}'},
])
key_values.to_csv(TAB / 'key_values_for_manuscript.csv', index=False)
display(key_values)

print(f'Population below 500 m, 2025: {bill(below500)} ({pct(below500 / global_pop_2025)})')
print(f'Population below 100 m, 2025: {bill(below100)}')
print(f'Population below 150 m, 2025: {bill(below150)} ({pct(below150_share)})')
print(f'Population-weighted median elevation, 2025: {population_weighted_median_elevation_2025:.0f} m')
print('Share below 150 m by age group, 2025: ' + ', '.join([
    f'0-14: {pct(below150_age_shares["young_0_14"])}',
    f'15-64: {pct(below150_age_shares["working_age_15_64"])}',
    f'65+: {pct(below150_age_shares["old_age_65_plus"])}',
]))
print(f'Population 1,500-3,500 m, 2025: {bill(highland_1500_3500)}')
print(f'Youth share below 500 m, 2025: {pct(youth_below500)}')
print(f'Youth share 1,500-3,500 m, 2025: {pct(youth_highland)}')
print(f'Growth below 500 m, 2015-2025: {pct(growth_below500)}')
print(f'Growth 1,500-3,500 m, 2015-2025: {pct(growth_highland)}')
print(f'Population >=3,500 m, 2025: {mill(highland_3500)} ({pct(highland_3500 / global_pop_2025)})')


## Robustness checks

The manuscript robustness summaries are stored in `data/dataset_s2_robustness_checks.csv`. They are generated from Dataset S1 by `processing/build_dataset_s2_robustness_checks.py` and do not reprocess the original gridded rasters. Dataset S2 is intended as a supporting dataset rather than an additional figure or SI table.
